# MoE-MedIR — SOTA Baselines Notebook

> **Conference**: ICARCV 2026

So sánh 4 SOTA image retrieval methods:

| Method | Venue | Backbone | Ý tưởng |
|---|---|---|---|
| **GeM** | TPAMI 2019 | ConvNeXt-Base (hardcoded) | GeM spatial pooling trên feature map [B,1024,H,W] |
| **DOLG** | ICCV 2021 | ConvNeXt-Base (hardcoded) | Global (GeM) + Local (attention) + orthogonal fusion |
| **CSQ** | CVPR 2020 | Tuỳ chọn ở Cell 4 | Binary hash codes 64-bit với class hash centers |
| **SuperGlobal** | ICCV 2023 | Kế thừa từ model gốc | α-QE re-ranking, không retrain |

**GeM + DOLG**: raw images → ConvNeXt-Base → spatial pooling. Không cần pre-extract features.  
**CSQ**: pre-extracted features. Cell 5 tự động extract nếu chưa có.  
**SuperGlobal**: post-processing trên embeddings của GeM/DOLG/CSQ sau khi train.  

**Setup**: GPU T4/P100 · Internet On

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    raise RuntimeError('No GPU — Settings → Accelerator → GPU')

## Cell 2 — Install Packages

In [ ]:
%%capture
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'medmnist', 'open_clip_torch', 'timm',
    'pytorch-metric-learning', 'scikit-learn',
    'matplotlib', 'seaborn'], check=True)
print('Done.')

## Cell 3 — Clone & Setup

In [ ]:
import os, subprocess

REPO_URL     = 'https://github.com/trong5nhan6/moe_medir.git'
PROJECT_ROOT = '/kaggle/working/moe_medir'

if os.path.exists(PROJECT_ROOT):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-3'])

## Cell 4 — ⚙️ Config

Config này áp dụng cho **CSQ** và **SuperGlobal** (dùng pre-extracted features).

> **GeM và DOLG** dùng `convnext_base` CNN hardcoded — không cần chỉnh backbone ở đây.

In [ ]:
import re, sys, importlib

# ════════════════════════════════════════════════════════════════
#  ✏️  CHỈNH SỬA TẠI ĐÂY  (chỉ ảnh hưởng CSQ + SuperGlobal)
#  GeM và DOLG dùng convnext_base CNN — không cần đổi ở đây
# ════════════════════════════════════════════════════════════════
BACKBONE     = "clip_vitb32"   # CSQ/SuperGlobal: phải khớp features đã extract
FEATURE_MODE = "concat"        # concat hoặc cls
EMBED_DIM    = 128
NUM_EXPERTS  = 8
BATCH_SIZE   = 256
EPOCHS       = 50
LR           = 1e-4
# ════════════════════════════════════════════════════════════════

def _patch(content, field, value):
    if isinstance(value, str):
        pattern     = rf'([ \t]+{field}\s*:\s*\w+\s*=\s*)["\'].*?["\']'
        replacement = rf'\g<1>"{value}"'
    else:
        pattern     = rf'([ \t]+{field}\s*:\s*[\w\[\]]+\s*=\s*)[\d.e+\-]+'
        replacement = rf'\g<1>{value}'
    new, n = re.subn(pattern, replacement, content)
    if n == 0:
        print(f"  [warn] field '{field}' not found")
    return new

with open('config.py', 'r') as f:
    cfg_text = f.read()
for field, val in [('backbone', BACKBONE), ('feature_mode', FEATURE_MODE),
                   ('embed_dim', EMBED_DIM), ('num_experts', NUM_EXPERTS),
                   ('batch_size', BATCH_SIZE), ('epochs', EPOCHS), ('lr', LR)]:
    cfg_text = _patch(cfg_text, field, val)
with open('config.py', 'w') as f:
    f.write(cfg_text)

if 'config' in sys.modules:
    import config as _c; importlib.reload(_c); from config import CFG
else:
    from config import CFG

print("=" * 55)
print(f"  backbone     : {CFG.backbone}  (CSQ / SuperGlobal)")
print(f"  feature_mode : {CFG.feature_mode}  →  feature_dim={CFG.feature_dim}")
print(f"  embed_dim    : {CFG.embed_dim}  epochs={CFG.epochs}  lr={CFG.lr}")
print("=" * 55)
print("  GeM / DOLG   : convnext_base (hardcoded, raw images)")

## Cell 5 — Features cho CSQ

**GeM và DOLG** dùng raw images trực tiếp — bỏ qua cell này.

Cell này dành cho **CSQ**: kiểm tra features theo backbone đã chọn ở Cell 4.  
Nếu chưa có → **tự động extract** (chạy `extract_features.py`), không cần làm gì thêm.

In [ ]:
import os, subprocess, sys
from config import CFG

SENTINEL = os.path.join(CFG.feature_dir, 'pathmnist_train_feat.npy')

if os.path.exists(SENTINEL):
    npy = [f for f in os.listdir(CFG.feature_dir) if f.endswith('.npy')]
    print(f'✅ Features sẵn sàng: {len(npy)} files tại {CFG.feature_dir}')
else:
    print(f'⚠️  Features chưa có tại {CFG.feature_dir}')
    print(f'   → Bắt đầu extract features cho backbone: {CFG.backbone} ...\n')
    subprocess.run([sys.executable, 'extract_features.py',
                    '--backbone', CFG.backbone], check=True)
    npy = [f for f in os.listdir(CFG.feature_dir) if f.endswith('.npy')]
    print(f'\n✅ Extract xong: {len(npy)} files tại {CFG.feature_dir}')

## Cell 6 — GeM (Generalized Mean Pooling) — TPAMI 2019

**Đúng paper gốc**: ConvNeXt-Base CNN → spatial feature map [B, 1024, 7×7] → GeM pooling.

```
feat_map = convnext.features(img)           # [B, 1024, 7, 7]
f_gem    = mean(feat_map^p)^(1/p)          # spatial GeM pooling
```

- `p` là tham số học được (khởi tạo = 3.0)
- `p=1` → GAP thông thường · `p→∞` → max pooling · `p~3` → sweet-spot

**2-Stage**: Stage 1 frozen backbone (5 epochs) → Stage 2 fine-tune toàn bộ ConvNeXt.  
~15–25 phút trên P100.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/gem_baseline.py'], check=True)

### Evaluate GeM — Test Set

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'eval/evaluate_sota.py',
                '--model', 'gem', '--split', 'test'], check=True)

## Cell 7 — DOLG (Dynamic Orthogonal Local-Global) — ICCV 2021

**Đúng paper gốc**: ConvNeXt-Base CNN → spatial feature map → 2 branches:

```
feat_map = convnext.features(img)           # [B, 1024, 7, 7]

Global: g = GeM(feat_map)                   # [B, 1024]
Local:  l = attention_conv(feat_map)        # [B, 1024] weighted spatial sum

Orthogonal fusion:
  l_orth = l - (l·ĝ)ĝ                      # loại bỏ phần l song song với g
  fused  = g + l_orth                       # chỉ giữ thông tin mới từ local
```

Local chỉ đóng góp thông tin **orthogonal** (không trùng) với global.

**2-Stage**: Stage 1 frozen backbone (5 epochs) → Stage 2 fine-tune toàn bộ ConvNeXt.  
~15–25 phút trên P100.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/dolg_baseline.py'], check=True)

### Evaluate DOLG — Test Set

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'eval/evaluate_sota.py',
                '--model', 'dolg', '--split', 'test'], check=True)

## Cell 8 — CSQ (Central Similarity Quantization)

**Ý tưởng**: Học binary hash codes (64-bit) cho retrieval:
- Mỗi class có một hash center cố định {-1,+1}^64
- Loss kéo features về hash center của class mình
- Retrieval: cosine similarity trên real-valued codes / Hamming distance trên binary codes

~5 phút trên T4.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/csq_baseline.py', '--n_bits', '64'], check=True)

### Evaluate CSQ — Test Set

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'eval/evaluate_sota.py',
                '--model', 'csq', '--split', 'test'], check=True)

## Cell 9 — SuperGlobal Re-ranking (α-QE)

**Ý tưởng**: Post-processing, không retrain. Áp dụng lên embeddings của GeM/DOLG/CSQ:
```
q' = normalize(q + Σ sim(q, gᵢ)^α × gᵢ)   for i in top-K neighbors
```
Sharpen query bằng cách aggregate top-K neighbors. Thường cải thiện +1~3% mAP@R.

- `alpha=3.0`: neighbors gần hơn được weight cao hơn
- `top_k=10`: aggregate 10 neighbors gần nhất

In [ ]:
import subprocess, sys, os

# SuperGlobal re-ranking trên test set cho các SOTA baselines đã train
for model_name in ['gem', 'dolg', 'csq']:
    ckpt = f'results/checkpoints/best_{model_name}.pt'
    if not os.path.exists(ckpt):
        print(f'⏭️  Bỏ qua {model_name} — chưa có checkpoint: {ckpt}')
        continue
    print(f'\n>>> SuperGlobal re-ranking: {model_name} (test set)')
    subprocess.run([
        sys.executable, 'baselines/superglobal.py',
        '--model', model_name,
        '--alpha', '3.0',
        '--top_k', '10',
        '--split', 'test',
    ], check=True)

## Cell 10 — So sánh kết quả

In [ ]:
import pandas as pd, os

# Test-set results từ evaluate_sota.py (lưu dạng results/test_{model}.csv)
test_files = {
    'GeM':  'results/test_gem.csv',
    'DOLG': 'results/test_dolg.csv',
    'CSQ':  'results/test_csq.csv',
}

rows = []
for name, path in test_files.items():
    if not os.path.exists(path):
        print(f'⏭️  {name}: chưa có ({path})')
        continue
    df  = pd.read_csv(path)
    avg = df.loc[df['Dataset'] == 'Average', 'mAP@R'].values
    if len(avg) == 0:
        continue
    row = {'Method': name, 'Test mAP@R': round(float(avg[0]), 2)}
    for _, r in df[df['Dataset'] != 'Average'].iterrows():
        row[r['Dataset'][:4]] = round(r['mAP@R'], 2)
    rows.append(row)

if rows:
    df_cmp = pd.DataFrame(rows).sort_values('Test mAP@R', ascending=False)
    print(df_cmp.to_string(index=False))
else:
    print('Chưa có kết quả. Chạy Cell 6 → 8 trước.')

## Cell 11 — Save Results

Tất cả file trong `/kaggle/working/` tự động lưu — tải về qua tab **Output**.

In [ ]:
import os
for root, dirs, files in os.walk('results'):
    for f in sorted(files):
        path = os.path.join(root, f)
        print(f'{path}  ({os.path.getsize(path)/1024:.1f} KB)')